# Unidade IV — Mineração de Padrões

## Itemsets frequentes e regras de associação

**Carga estimada:** 3 horas  
**Pré-requisitos:** conjuntos, probabilidade condicional e pandas.

> **Pergunta norteadora:** como identificar itens que aparecem juntos sem confundir frequência, capacidade de previsão e causalidade?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- representar transações como conjuntos e matriz *one-hot*;
- definir itemset, suporte e regra de associação;
- calcular suporte, confiança e *lift* manualmente;
- gerar e filtrar regras sem atribuir causalidade indevida.


In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


## Dados transacionais

Cada transação é um conjunto de itens, sem quantidade nem ordem. A matriz *one-hot* possui uma linha por compra e uma coluna booleana por item. Essa representação responde apenas se o item ocorreu; sequência e quantidade exigem outras estruturas.


In [2]:
transacoes = [
    ["arroz", "feijao", "oleo"], ["arroz", "feijao"],
    ["arroz", "leite"], ["pao", "leite", "manteiga"],
    ["pao", "leite"], ["arroz", "feijao", "oleo"],
    ["pao", "cafe"], ["arroz", "feijao", "leite"],
    ["pao", "leite", "manteiga"], ["arroz", "feijao"],
    ["cafe", "leite"], ["arroz", "feijao", "oleo", "leite"],
]
codificador = TransactionEncoder()
matriz = codificador.fit(transacoes).transform(transacoes)
cestas = pd.DataFrame(matriz, columns=codificador.columns_)
cestas.index = pd.Index(range(1, len(cestas) + 1), name="transacao")
cestas


,arroz,cafe,feijao,leite,manteiga,oleo,pao
transacao,,,,,,,
1,True,False,True,False,False,True,False
2,True,False,True,False,False,False,False
3,True,False,False,True,False,False,False
4,False,False,False,True,True,False,True
5,False,False,False,True,False,False,True
6,True,False,True,False,False,True,False
7,False,True,False,False,False,False,True
8,True,False,True,True,False,False,False
9,False,False,False,True,True,False,True


## Suporte, confiança e lift

Para itemsets disjuntos $X$ e $Y$ e $N$ transações:

$$\operatorname{sup}(X)=\frac{\#(X)}{N},$$

$$\operatorname{conf}(X\rightarrow Y)=\frac{\operatorname{sup}(X\cup Y)}{\operatorname{sup}(X)},$$

$$\operatorname{lift}(X\rightarrow Y)=\frac{\operatorname{conf}(X\rightarrow Y)}{\operatorname{sup}(Y)}.$$

Suporte mede prevalência conjunta. Confiança é a proporção das ocorrências de $X$ que também contêm $Y$. *Lift* compara essa confiança à prevalência de $Y$: acima de 1 indica associação positiva, perto de 1 sugere independência e abaixo de 1 indica associação negativa na amostra.


In [3]:
def suporte(itens: set[str], compras: list[list[str]]) -> float:
    """Calcula a fração de transações que contêm todos os itens."""
    return sum(itens.issubset(set(compra)) for compra in compras) / len(compras)


x, y = {"feijao"}, {"arroz"}
suporte_x = suporte(x, transacoes)
suporte_y = suporte(y, transacoes)
suporte_xy = suporte(x | y, transacoes)
confianca = suporte_xy / suporte_x
lift = confianca / suporte_y
pd.Series({
    "suporte(feijao)": suporte_x, "suporte(arroz)": suporte_y,
    "suporte(feijao, arroz)": suporte_xy,
    "confianca(feijao -> arroz)": confianca,
    "lift(feijao -> arroz)": lift,
}).round(3)


suporte(feijao)               0.500
suporte(arroz)                0.583
suporte(feijao, arroz)        0.500
confianca(feijao -> arroz)    1.000
lift(feijao -> arroz)         1.714
dtype: float64

Seis das doze compras contêm feijão e todas essas seis também contêm arroz; a confiança é 1. Como arroz aparece em $7/12$ das compras, o *lift* é aproximadamente 1,714. Isso descreve coocorrência na amostra: não prova que feijão causa a compra de arroz nem que uma intervenção aumentará vendas.


## Mineração e filtragem

Primeiro encontramos itemsets que atendem ao suporte mínimo; depois geramos regras. Limiares são decisões do problema: suporte alto pode ocultar nichos, enquanto suporte muito baixo pode produzir coincidências instáveis.


In [4]:
itemsets_frequentes = apriori(cestas, min_support=0.15, use_colnames=True)
regras = association_rules(
    itemsets_frequentes, metric="confidence", min_threshold=0.60
)
regras_selecionadas = (
    regras.loc[:, ["antecedents", "consequents", "support", "confidence", "lift"]]
    .sort_values(["lift", "support"], ascending=False)
)
regras_selecionadas.head(10).round(3)


,antecedents,consequents,support,confidence,lift
13,"(leite, pao)",(manteiga),0.167,0.667,4.000
15,(manteiga),"(leite, pao)",0.167,1.000,4.000
6,(manteiga),(pao),0.167,1.000,3.000
12,"(leite, manteiga)",(pao),0.167,1.000,3.000
3,(oleo),(feijao),0.250,1.000,2.000
9,"(arroz, oleo)",(feijao),0.250,1.000,2.000
11,(oleo),"(arroz, feijao)",0.250,1.000,2.000
0,(arroz),(feijao),0.500,0.857,1.714
1,(feijao),(arroz),0.500,1.000,1.714
2,(oleo),(arroz),0.250,1.000,1.714


Regras invertidas têm o mesmo suporte e *lift*, mas podem ter confianças diferentes. Antes de agir, verifique volume, estabilidade, redundância, custos, disponibilidade e explicações alternativas, como promoções e sazonalidade.

> **U04-NB01-V01 — Verifique seu entendimento:** por que uma regra com confiança de 90% pode ter *lift* menor que 1?

> **U04-NB01-E01 — Exercício:** calcule manualmente suporte, confiança e *lift* para `pao -> leite` e para a regra inversa. Compare as confianças e explique por que o *lift* é igual nas duas direções.


## Síntese

- Dados transacionais representam presença de itens por evento.
- Suporte mede prevalência; confiança é condicional; *lift* compara com a taxa-base.
- Regras fortes podem ser raras, redundantes, instáveis ou sem utilidade.
- Associação observada não demonstra causalidade.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 4, seções 4.1–4.2.
- RASCHKA, Sebastian. *mlxtend documentation*: frequent patterns. Versão 0.23.
